# T7 Arm B: score F0 / F1 / F2 on Paper 1's seven-model ladderScoring only. **No Arm B statistic is computed here**; analysis happens in aseparate session against the preregistration.One concern per cell, so a failure is locatable. The launch is staged: cell 3runs the smallest model, cell 4 prints the F0 verdict and **halts**, and cell 5is behind a flag that defaults to off so a Run All cannot skip the verdict.Everything is driven from `kaggle_t7_framings.py`, which is the entry point. Thisnotebook does not re-implement any of it, and in particular does not bypassD102 part 3 (`size` at batch 1), D104, D109, D111 or D117.**Procedure, paths and what to bring back: `docs/P2/KAGGLE_T7.md`.**

## 1. Environment captureWritten to `env_t7.json` **before any model loads**, so it survives a session that dies during the first download. Pinned commits are asserted against Paper 1's own `env.json` when the P1 dataset is mounted.

In [ ]:
import os, sys, jsonsys.path.insert(0, "/kaggle/input/deception-p2/notebooks")# Paths. Override here if your dataset slugs differ; see docs/P2/KAGGLE_T7.md.os.environ.setdefault("P2_ROOT",   "/kaggle/input/deception-p2")os.environ.setdefault("P1_ROOT_K", "/kaggle/input/deception-p1")os.environ.setdefault("P1_SRC",    "/kaggle/input/deception-p1/src")os.environ.setdefault("P2_SRC",    "/kaggle/input/deception-p2/src")import kaggle_t7_framings as KENV = K.capture_env()          # raises if a pin disagrees with P1's recordprint(f"torch {ENV['torch']}  transformers {ENV['transformers']}  "      f"tokenizers {ENV['tokenizers']}  accelerate {ENV['accelerate']}")print(f"cuda {ENV['cuda_version']}  cudnn {ENV['cudnn']}  "      f"devices {ENV['device_count']}")for d in ENV["devices"]:    print(f"  [{d['index']}] {d['name']}  cc{d['capability']}  "          f"{d['free_MiB']:,} / {d['total_MiB']:,} MiB free")print()for k, v in ENV["models"].items():    print(f"  {k:5s} {v['repo']:34s} {v['revision'][:12]}  {v['family']}")x = ENV["p1_revision_crosscheck"]print(f"\npin cross-check: {x['note']}  ({x['n_checked']} checked, "      f"{len(x['problems'])} problems)")assert ENV["cuda_available"], "no GPU. Settings -> Accelerator -> GPU T4 x2"print(f"\nenv_t7.json written to {K.OUTDIR}")

## 2. Input verificationSHA256 of `items_final.parquet` against T6's gate record, and of `t7_stimuli.parquet` against the manifest the rendering session wrote. **A stale dataset upload is silent otherwise**: the run completes and the numbers are wrong.

In [ ]:
import pandas as pdver = K.verify_inputs()        # raises on any mismatch or missing hashfor label, v in ver.items():    print(f"  {label:24s} {v['sha256'][:16]}  {'MATCHES' if v['matches'] else 'MISMATCH'}")K.assert_no_sampling()         # P2-D9: argmax over teacher-forced log-probsprint(f"  no-sampling check        {K.ENV['no_sampling_check']}")S     = pd.read_parquet(K.STIMULI)TILES = {t["id"]: t for t in json.load(open(K.TILES_JSON))["tiles"]}ROWS  = pd.read_parquet(K.ITEMS)print(f"\n{len(S):,} renderings, {S['item_id'].nunique():,} items, "      f"framings {sorted(S['framing'].unique())}")print(f"expected rows per model: {len(S) * K.N_RULES:,}   "      f"per (model, tile): {K.expected_rows(S, 'size'):,}")_, missing = K.preflight_models()assert not missing, "see the message above; fix the model source before running"

## 2b. Resume drillExercised once, here, before any GPU time is spent. Writes a short checkpoint and an unreadable one, and requires both to be caught and re-queued while a complete one is reused. **A checkpoint scheme that has never resumed is an assumption.**

In [ ]:
K.simulate_kill_and_resume(S)# What this session will actually do, given what is already on disk.plan = K.plan_run(S)print(f"\nestimated GPU-hours remaining: "      f"{sum(K.ESTIMATE_HOURS[m] for m in {m for m, _ in plan['todo']}):.1f} "      f"of {sum(K.ESTIMATE_HOURS.values()):.1f} for the full ladder")

## 3. Stage 1: smallest ladder model only`L1` (Qwen3-0.6B), all three framings, all 1,000 items. Runs and finishes without stage 2 existing in the session. Each `(model, tile)` is written to `/kaggle/working` the moment it completes.

In [ ]:
STAGE1 = ["L1"]K.determinism_gate("L1", S, TILES)      # bit-identical re-score, or stopplan1 = K.plan_run(S, models=STAGE1)D1 = K.run_rung("L1", S, TILES, ROWS, done_already=set(plan1["done"]))print(f"\nstage 1 rows: {len(D1):,} of {len(S) * K.N_RULES:,} expected")

## 4. Stage 1 verdict: the F0 disagreement count`F0` is Paper 1's `cond4` prompt byte for byte (P2-D1), so a disagreement is pure environment noise: identical text, different run. This is P2-D13's measured noise floor, and the first empirical measurement of the no-effect rate `PREREGISTRATION_v2.6.md` section 1.1 could only argue from the code path.**Stop here and read it.** Do not run cell 5 until you have.

In [ ]:
ALL1 = pd.read_parquet(os.path.join(K.CKPT, "t7_L1.parquet"))P1_CHOICES = [os.path.join(K.P1_NOTEBOOKS, "results_2/choices_llm.parquet"),              os.path.join(K.P1_NOTEBOOKS, "results_3/choices_llm_t26.parquet")]counts = K.f0_disagreement(ALL1, P1_CHOICES)K.ENV["f0_vs_cond4"] = countsfloor = K.implied_floor(counts)K.ENV["implied_floor"] = floorjson.dump(K.ENV, open(os.path.join(K.OUTDIR, "env_t7.json"), "w"), indent=2, default=str)print("=" * 72)print("STAGE 1 VERDICT  F0 versus Paper 1's frozen cond4")print("=" * 72)if counts is None:    print("P1 choices not mounted. Bring choices_t7.parquet back and run the")    print("comparison locally; the floor is set there instead.")else:    print(f"{'model':6s} {'tile':9s} {'rend':>7s} {'disagree':>9s} "          f"{'items':>6s} {'items dis':>10s}")    for m, per_tile in counts.items():        for t, v in sorted(per_tile.items()):            print(f"{m:6s} {t:9s} {v['renderings']:7,d} "                  f"{v['disagreeing_renderings']:9,d} {v['items']:6,d} "                  f"{v['disagreeing_items']:10,d}")    print()    for m, v in floor["per_model"].items():        print(f"  {m}: {v['disagreeing_items']} of {v['items']} size-tile items "              f"disagree -> floor = max({floor['provisional_floor']}, "              f"{v['disagreeing_items']}) = {v['implied_floor']}")        print(f"     context: P1's own batch-composition flip rate for {m} was "              f"{v['p1_batch_flip_rate']:.3f} on the non-size tiles,")        print(f"     and P1 scores size at batch 1 by design, as this run does.")    print(f"\n  {floor['caveat']}")    print(f"  rule: {floor['rule']}")print("=" * 72)print("HALT. Read the count above before running stage 2.")print("A count far above P1's flip rates means this environment is noisier than")print("the one that produced cond4, and the floor rises with it. That is the")print("measurement doing its job, not a failure.")print("=" * 72)

## 5. Stage 2: the remaining six modelsBehind a flag that **defaults to off**, so a Run All stops at the verdict. Set `RUN_STAGE2 = True` deliberately, after reading cell 4.An OOM on one `(model, tile)` is recorded and the run continues to the next; completed tiles are already on disk.

In [ ]:
RUN_STAGE2 = False      # <- set True only after reading the stage 1 verdictSTAGE2 = ["L2", "L3", "B2", "CTRL", "L4", "B4"]   # ascending costif not RUN_STAGE2:    print("stage 2 not enabled. Set RUN_STAGE2 = True to run:")    for m in STAGE2:        print(f"  {m:5s} {K.MODELS[m][0]:34s} ~{K.ESTIMATE_HOURS[m]:.2f} h")    print(f"  total ~{sum(K.ESTIMATE_HOURS[m] for m in STAGE2):.1f} GPU-hours")else:    plan2 = K.plan_run(S, models=STAGE2)    for m in STAGE2:        K.run_rung(m, S, TILES, ROWS, done_already=set(plan2["done"]))        json.dump(K.ENV, open(os.path.join(K.OUTDIR, "env_t7.json"), "w"),                  indent=2, default=str)    if K.ENV.get("failures"):        print(f"\n{len(K.ENV['failures'])} cell(s) failed and were skipped:")        for f in K.ENV["failures"]:            print(f"  {f['model']:5s} {f['tile']:8s} {f['error']}")

## 6. Export`choices_t7.parquet` and `env_t7.json` to `/kaggle/working`, assembled from the checkpoints on disk rather than from memory, with a completeness check per model. Safe to run after a partial session: it reports what is missing rather than truncating silently.

In [ ]:
ALL = K.export(S)print()used = K.ENV.get("model_hours", {})if used:    print(f"{'model':6s} {'used h':>8s} {'est h':>7s} {'ratio':>7s}")    for m, h in used.items():        e = K.ESTIMATE_HOURS[m]        print(f"{m:6s} {h:8.2f} {e:7.2f} {h / e:7.2f}x")    print(f"{'TOTAL':6s} {sum(used.values()):8.2f} "          f"{sum(K.ESTIMATE_HOURS.values()):7.2f}")print("\nBring back from /kaggle/working:")print("  choices_t7.parquet   the scored choices")print("  env_t7.json          environment, pins, F0 counts, budget, failures")print("\nAnalysis happens in a separate session against the preregistration.")print("Nothing in this notebook computed an Arm B statistic.")